# Control the arm by **text**, **voice** or **hand gestures**

One notebook, one panel. At the end you are asked **how you want to give instructions**; only the method you pick is started
(so missing voice/vision packages never break the text mode).

```
 THIS NOTEBOOK (control panel)                              sim_server.py  (separate process, started for you)
 ┌──────────────────────────────────┐   UDP "hammer"     ┌───────────────────────────────────────────────┐
 │ Text box / buttons               │ ─────────────────> │ MuJoCo viewer + SmolVLA policy at 20 Hz        │
 │ Microphone + Whisper             │                    │ idle -> run -> success -> home -> idle         │
 │ Camera + MediaPipe hand gestures │ <───────────────── │ prints [state] / [event] lines (shown in log)  │
 └──────────────────────────────────┘      stdout        └───────────────────────────────────────────────┘
```

**Why two processes?** The MuJoCo window needs its own main thread, and a notebook cell that runs a long loop would freeze the buttons.

**What the simulation process guarantees** (these are the same rules as `deploy_policy.ipynb`):
- it waits for a command, it never starts moving on its own;
- a command that arrives while the arm is busy is ignored (so a gesture held for a long time cannot queue extra picks);
- a tool that is already in the green tray is refused;
- after every pick the arm returns to the home pose, because every training episode started there.

Run the cells from top to bottom. Start Jupyter from the repo root (the folder that contains `asset/`, `mujoco_env/` and `demo_data_garage/`).

## 1. Settings  (the only cell you normally edit)

In [ ]:
import os, sys
sys.path.insert(0, os.getcwd())

# ---------------- simulation and policy ----------------
CKPT         = "./ckpt/smolvla_garage/checkpoints/025000/pretrained_model"   # the folder that contains config.json + model.safetensors
DATASET_REPO = "garage_pnp"
DATASET_ROOT = "./demo_data_garage"      # only its meta/ folder is read, so you may delete data/ to save disk
XML          = "./asset/example_scene_tools.xml"
UDP_PORT     = 5005                      # close deploy_policy.ipynb first: it uses the same port
SIM_SCRIPT   = "sim_server.py"           # written by the next cell
HZ, MAX_RUN_TICKS, MIN_RUN_TICKS, HOME_TICKS = 20, 600, 30, 60
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

# ---------------- voice ----------------
WHISPER_MODEL    = "base"         # tiny / base / small  (bigger = more accurate, slower, more CPU shared with the policy)
WHISPER_PROMPT   = "Pick up the wrench, screwdriver, hammer, or pliers."    # helps Whisper spell the tool names
END_SILENCE_SEC  = 0.8            # a phrase ends after this much silence
MAX_PHRASE_SEC   = 6.0
MIN_SPEECH_RMS   = 0.01           # never treat anything quieter than this as speech
NOISE_MARGIN     = 3.0            # speech must be this many times louder than the room noise measured at start

# ---------------- vision ----------------
CAMERA_INDEX     = 0              # 0 is usually the built-in webcam
GESTURE_TO_TOOL  = {"1": "wrench", "2": "screwdriver", "3": "hammer", "4": "pliers"}   # fingers held up -> tool
HOLD_SEC, GRACE_SEC       = 0.6, 0.15     # hold a gesture this long; glitches shorter than GRACE_SEC are forgiven
COOLDOWN_SEC, REARM_SEC   = 3.0, 1.0      # min seconds between two sends; relax your hand this long to re-arm
REQUIRE_STILL, MOTION_MAX_PER_SEC = True, 0.8   # a moving hand does not count

print("device:", DEVICE, "| checkpoint:", CKPT)

## 2. The simulation process
This cell **writes** `sim_server.py` next to the notebook; it does not run it. The panel starts it for you.
It is the validated loop from `deploy_policy.ipynb`, plus a UDP command channel. External programs (for example
`gesture_command.py` or `send_command.py`) can send the same plain-text commands to the same port.

In [ ]:
%%writefile sim_server.py
"""
sim_server.py - the simulation process: MuJoCo viewer + SmolVLA policy, controlled over UDP.

The control-panel notebook starts this program for you. To run it by hand:
    python sim_server.py --ckpt ./ckpt/smolvla_garage/checkpoints/020000/pretrained_model

Commands (UDP, plain text, localhost):
    wrench | screwdriver | hammer | pliers      (or any sentence that names exactly ONE tool)
    reset                                       reshuffle the tools back into the source tray
    quit

Status lines it prints (the notebook reads them):
    [state] idle | run:<tool> | home
    [event] success:<tool> | timeout:<tool> | refused:<tool> | reset
"""
import argparse
import os
import queue
import re
import socket
import sys
import threading
import time

import numpy as np

# ------------------------------------------------------------------ command parsing (no heavy imports)
TOOL_SYNONYMS = {
    "wrench": ["wrench", "spanner"],
    "screwdriver": ["screwdriver", "screw driver"],
    "hammer": ["hammer"],
    "pliers": ["pliers", "plier", "plyers"],
}


def tool_from_text(text):
    """The tool named in `text`, or None if no tool or more than one tool is named."""
    t = text.lower()
    hits = [tool for tool, words in TOOL_SYNONYMS.items()
            if any(re.search(r"\b" + re.escape(w) + r"\b", t) for w in words)]
    return hits[0] if len(hits) == 1 else None


def parse_command(text):
    """-> ('tool', name) | ('reset', None) | ('quit', None) | (None, None)"""
    t = " ".join(text.strip().lower().split())
    if t.startswith("task\t"):                       # old notebook protocol
        t = t[5:].strip()
    if t in ("quit", "exit"):
        return ("quit", None)
    tool = tool_from_text(t)
    if tool:
        return ("tool", tool)
    if re.search(r"\breset\b", t):
        return ("reset", None)
    return (None, None)


# ------------------------------------------------------------------ the state machine (no IO of its own)
class ArmController:
    """idle -> run (policy drives the arm) -> home (scripted return) -> idle.
    * a command that arrives while the arm is busy is ignored (gesture/voice programs can repeat themselves)
    * a tool that is already in the green tray is refused
    * every run starts from the home pose, exactly like the training episodes"""

    def __init__(self, reset_policy, max_run_ticks=600, min_run_ticks=30, home_ticks=60):
        self.reset_policy = reset_policy
        self.max_run_ticks, self.min_run_ticks, self.home_ticks = max_run_ticks, min_run_ticks, home_ticks
        self.mode, self.tool, self.t, self.home_from = "idle", None, 0, None

    def _state_line(self):
        return f"[state] run:{self.tool}" if self.mode == "run" else f"[state] {self.mode}"

    def _start_home(self, env):
        self.mode, self.t, self.home_from = "home", 0, env.get_action_vec()

    def tick(self, env, get_action, commands, emit=print):
        """one 20 Hz control step. returns 'quit' to stop the program, else None."""
        for kind, arg in commands:
            if kind is None:
                emit("[event] unknown command (name exactly one tool: wrench, screwdriver, hammer, pliers)")
            elif kind == "quit":
                return "quit"
            elif kind == "reset":
                env.reset()
                env.set_instruction("")
                self.reset_policy()
                self.mode, self.tool, self.t = "idle", None, 0
                emit("[event] reset")
            elif kind == "tool":
                if self.mode != "idle":
                    emit(f"[event] busy - ignored '{arg}'")
                elif env.tool_in_tray(arg, "dst"):
                    emit(f"[event] refused:{arg} (it is already in the green tray; send 'reset' to reshuffle)")
                else:
                    from mujoco_env.y_env_tools import PROMPT          # imported lazily so tests need no lerobot
                    self.mode, self.tool, self.t = "run", arg, 0
                    env.set_instruction(PROMPT.format(arg))
                    self.reset_policy()                                # clear the previous action chunk
                    emit(f"[event] start:{arg} -> \"{env.instruction}\"")
            emit(self._state_line())

        if self.mode == "run":
            action = np.asarray(get_action(env, env.instruction), dtype=np.float64)
            if not np.isfinite(action).all():
                emit("[event] non-finite action; going home")
                self._start_home(env)
                emit(self._state_line())
                return None
            env.step(action)
            self.t += 1
            if self.t >= self.min_run_ticks and env.check_success(self.tool):
                emit(f"[event] success:{self.tool}")
                self._start_home(env)
                emit(self._state_line())
            elif self.t >= self.max_run_ticks:
                emit(f"[event] timeout:{self.tool} (send 'reset' if it is stuck)")
                self._start_home(env)
                emit(self._state_line())
        elif self.mode == "home":
            self.t += 1
            a = min(1.0, self.t / self.home_ticks)
            vec = (1 - a) * self.home_from + a * env.home_action_vec()
            vec[7] = 0.0                                               # open the gripper
            env.step(vec)
            if self.t >= self.home_ticks + 10:
                self.mode, self.tool = "idle", None
                env.set_instruction("")
                emit(self._state_line())
        return None


# ------------------------------------------------------------------ UDP
def open_socket(host, port):
    s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)               # no SO_REUSEADDR: a second server must fail loudly
    s.bind((host, port))
    s.settimeout(0.5)
    return s


def udp_listener(sock, q, stop):
    while not stop.is_set():
        try:
            data, _ = sock.recvfrom(4096)
        except socket.timeout:
            continue
        except OSError:
            break
        q.put(data.decode("utf-8", errors="ignore"))


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--ckpt", required=True, help=".../checkpoints/<step>/pretrained_model")
    p.add_argument("--dataset_repo", default="garage_pnp")
    p.add_argument("--dataset_root", default="./demo_data_garage", help="only meta/ is read")
    p.add_argument("--xml", default="./asset/example_scene_tools.xml")
    p.add_argument("--device", default="cpu")
    p.add_argument("--hz", type=int, default=20)
    p.add_argument("--host", default="127.0.0.1")
    p.add_argument("--port", type=int, default=5005)
    p.add_argument("--max_run_ticks", type=int, default=600)
    p.add_argument("--min_run_ticks", type=int, default=30)
    p.add_argument("--home_ticks", type=int, default=60)
    p.add_argument("--parent_pid", type=int, default=0, help="exit when this process dies (set by the notebook)")
    return p.parse_args()


def main():
    a = parse_args()
    q, stop = queue.Queue(), threading.Event()
    try:
        sock = open_socket(a.host, a.port)                              # fail fast, before the slow policy load
    except OSError as exc:
        print(f"[error] cannot listen on {a.host}:{a.port} ({exc}). Another simulation (or deploy_policy.ipynb) "
              f"is probably still running.", flush=True)
        sys.exit(2)
    threading.Thread(target=udp_listener, args=(sock, q, stop), daemon=True).start()

    for f in ("config.json", "model.safetensors"):
        if not os.path.exists(os.path.join(a.ckpt, f)):
            print(f"[error] {a.ckpt}/{f} not found. Point CKPT at the 'pretrained_model' folder of a checkpoint.", flush=True)
            sys.exit(2)

    print("loading the policy (this can take a minute)...", flush=True)
    import torch
    from PIL import Image
    from torchvision import transforms
    from lerobot.common.datasets.lerobot_dataset import LeRobotDatasetMetadata
    from lerobot.configs.policies import PreTrainedConfig
    from lerobot.common.policies.smolvla.modeling_smolvla import SmolVLAPolicy
    from mujoco_env.y_env_tools import SimpleEnvTools

    meta = LeRobotDatasetMetadata(a.dataset_repo, root=a.dataset_root)
    cfg = PreTrainedConfig.from_pretrained(a.ckpt)
    cfg.device = a.device
    policy = SmolVLAPolicy.from_pretrained(a.ckpt, config=cfg, dataset_stats=meta.stats).to(a.device).eval()
    to_tensor = transforms.ToTensor()

    def get_action(env, task):
        rgb, wrist = env.grab_image()
        f = lambda img: to_tensor(Image.fromarray(img).resize((256, 256))).unsqueeze(0).to(a.device)
        obs = {"observation.state": torch.tensor(env.get_joint_state()).unsqueeze(0).to(a.device),   # 8-D
               "observation.image": f(rgb), "observation.wrist_image": f(wrist), "task": [task]}
        with torch.no_grad():
            return policy.select_action(obs)[0, :8].cpu().numpy()                                    # 6 joints, slide, gripper

    env = SimpleEnvTools(a.xml, action_type="joint_angle", seed=0)
    ctrl = ArmController(policy.reset, a.max_run_ticks, a.min_run_ticks, a.home_ticks)
    policy.reset()
    print(f"READY. listening on {a.host}:{a.port}", flush=True)
    print("[state] idle", flush=True)

    emit = lambda s: print(s, flush=True)
    try:
        while env.env.is_viewer_alive():
            if a.parent_pid and os.getppid() != a.parent_pid:           # the notebook is gone: do not leave an orphan window
                break
            env.step_env()                                              # physics at 500 Hz
            if not env.env.loop_every(HZ=a.hz):
                continue                                                # control at 20 Hz
            commands = []
            while not q.empty():
                commands.append(parse_command(q.get()))
            if ctrl.tick(env, get_action, commands, emit) == "quit":
                break
            if ctrl.mode != "run":
                env.grab_image()                                        # keep the overlay cameras fresh
            env.render()
    except KeyboardInterrupt:
        pass
    finally:
        stop.set()
        sock.close()
        try:
            env.env.close_viewer()
        except Exception:
            pass
        print("[state] stopped", flush=True)


if __name__ == "__main__":
    main()

## 3. Helpers: starting the simulation and checking requirements
`preflight()` checks everything the chosen method needs **before** anything starts, and tells you exactly what to install or fix.

In [ ]:
import atexit, collections, html, importlib, importlib.util, queue, re, socket, subprocess, threading, time
import numpy as np

import sim_server
importlib.reload(sim_server)
from sim_server import parse_command

_log_sink = {"fn": print}                      # the control panel plugs its log window in here
def log(message):
    _log_sink["fn"](message)


def preflight(method=None, check_common=True):
    """-> (problems, warnings). Nothing is started if there are problems."""
    problems, warnings = [], []
    if check_common:
        for f in ("config.json", "model.safetensors"):
            if not os.path.exists(os.path.join(CKPT, f)):
                problems.append(f"CKPT: '{CKPT}/{f}' not found. Set CKPT to the 'pretrained_model' folder of a checkpoint.")
        if not os.path.exists(os.path.join(DATASET_ROOT, "meta", "info.json")):
            problems.append(f"DATASET_ROOT: '{DATASET_ROOT}/meta/info.json' not found (the policy needs the dataset statistics).")
        if not os.path.exists(XML):
            problems.append(f"XML: '{XML}' not found.")
        if not os.path.isdir("mujoco_env"):
            problems.append("The folder 'mujoco_env' is not here: start Jupyter from the repo root.")
        if not os.path.exists(SIM_SCRIPT):
            problems.append(f"'{SIM_SCRIPT}' not found: run the cell that writes it (the one starting with %%writefile).")
        try:
            s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM); s.bind(("127.0.0.1", UDP_PORT)); s.close()
        except OSError:
            problems.append(f"UDP port {UDP_PORT} is in use: close deploy_policy.ipynb / an old sim_server.py "
                            f"(terminal: pkill -f sim_server.py), or change UDP_PORT.")

    if method == "voice":
        for module, hint in (("sounddevice", "pip install sounddevice   (Ubuntu also: sudo apt install libportaudio2)"),
                             ("whisper", "pip install openai-whisper")):
            try:
                importlib.import_module(module)
            except Exception as exc:
                problems.append(f"voice needs '{module}' ({type(exc).__name__}). Install: {hint}")
        else:
            try:
                import sounddevice as sd
                if not any(d["max_input_channels"] > 0 for d in sd.query_devices()):
                    problems.append("no microphone found by sounddevice.")
            except Exception:
                pass

    if method == "vision":
        for module, hint in (("cv2", "pip install opencv-python-headless"), ("mediapipe", "pip install mediapipe")):
            try:
                importlib.import_module(module)
            except Exception as exc:
                problems.append(f"vision needs '{module}' ({type(exc).__name__}). Install: {hint}")
        for f in ("gesture.py", "hand_landmarker.task"):
            if not os.path.exists(f):
                problems.append(f"vision needs '{f}' in the repo root.")
        if not problems:
            try:
                import cv2, gesture as G
                if dict(G.GESTURE_TO_TOOL) != dict(GESTURE_TO_TOOL):
                    warnings.append(f"GESTURE_TO_TOOL here {GESTURE_TO_TOOL} differs from gesture.py {G.GESTURE_TO_TOOL}; this notebook's table is used.")
                cap = cv2.VideoCapture(CAMERA_INDEX, G.CAMERA_BACKEND)
                if not cap.isOpened():
                    problems.append(f"camera {CAMERA_INDEX} could not be opened (try another CAMERA_INDEX; close other programs using the camera).")
                cap.release()
            except Exception as exc:
                problems.append(f"vision check failed: {exc!r}")
    return problems, warnings


class SimProcess:
    """Starts sim_server.py, forwards its output to the log, tracks its state, sends it commands over UDP."""

    def __init__(self, on_state=None, script=None):
        self.script = script or SIM_SCRIPT
        self.proc, self.state, self.on_state = None, "stopped", on_state
        self.sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

    @property
    def running(self):
        return self.proc is not None and self.proc.poll() is None

    def _set_state(self, state):
        self.state = state
        if self.on_state:
            self.on_state(state)

    def start(self):
        cmd = [sys.executable, "-u", self.script, "--ckpt", CKPT, "--dataset_repo", DATASET_REPO,
               "--dataset_root", DATASET_ROOT, "--xml", XML, "--device", DEVICE, "--hz", str(HZ),
               "--port", str(UDP_PORT), "--max_run_ticks", str(MAX_RUN_TICKS), "--min_run_ticks", str(MIN_RUN_TICKS),
               "--home_ticks", str(HOME_TICKS), "--parent_pid", str(os.getpid())]
        self._set_state("starting")
        self.proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=os.getcwd())
        threading.Thread(target=self._pump, args=(self.proc,), daemon=True).start()

    def _pump(self, proc):
        for line in proc.stdout:
            line = line.rstrip()
            m = re.match(r"\[state\] (.*)", line)
            if m:
                self._set_state(m.group(1).strip())
            log("[sim] " + line)
        code = proc.wait()
        self._set_state("stopped")
        log(f"[sim] process ended (exit code {code})")

    def mark_busy(self):
        """block further commands until the simulation reports its real state"""
        if self.state == "idle":
            self._set_state("busy")

    def send(self, text):
        self.sock.sendto(text.encode("utf-8"), ("127.0.0.1", UDP_PORT))

    def stop(self):
        if self.proc is None:
            return
        if self.running:
            try:
                self.send("quit")
                self.proc.wait(timeout=10)
            except Exception:
                self.proc.terminate()
                try:
                    self.proc.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    self.proc.kill()

## 4. The three input methods
All three end in the same place: they send **one tool name** to the simulation.

- **Voice:** listens continuously; detects when you start and stop speaking (no fixed recording windows), measures the room noise
  first, transcribes with Whisper, and acts only if the sentence names **exactly one tool**. Speech heard while the arm is working is ignored,
  which also keeps the CPU free for the policy.
- **Vision:** your `gesture.py` recognizer. Hold 1-4 fingers still for `HOLD_SEC`; one gesture sends one command, then you must relax your hand to re-arm.
  Gestures shown while the arm is working are ignored.
- **Text:** a text box and buttons (in the panel below).

In [ ]:
def _rms(x):
    return float(np.sqrt(np.mean(np.square(x))))

_WHISPER_CACHE = {}


class VoiceInput:
    def __init__(self, on_command, set_status, get_sim_state):
        self.on_command, self.set_status, self.get_sim_state = on_command, set_status, get_sim_state
        self.thread, self.stop_event = None, threading.Event()

    def start(self, mic, model_name):
        if self.thread is not None and self.thread.is_alive():
            return
        self.stop_event = threading.Event()
        self.thread = threading.Thread(target=self._run, args=(mic, model_name, self.stop_event), daemon=True)
        self.thread.start()

    def stop(self):
        self.stop_event.set()

    def _run(self, mic, model_name, stop_event):
        try:
            import sounddevice as sd
            import whisper
        except Exception as exc:
            log(f"[voice] cannot start: {exc!r}. Install: pip install openai-whisper sounddevice  (Ubuntu: sudo apt install libportaudio2)")
            self.set_status("Voice unavailable (see log)")
            return
        try:
            if model_name not in _WHISPER_CACHE:
                self.set_status(f"Loading Whisper '{model_name}' (first time only)...")
                _WHISPER_CACHE[model_name] = whisper.load_model(model_name)
            model = _WHISPER_CACHE[model_name]

            sr = 16000
            block = int(0.1 * sr)                                  # 100 ms blocks
            end_blocks = max(1, int(END_SILENCE_SEC / 0.1))
            max_blocks = int(MAX_PHRASE_SEC / 0.1)
            q = queue.Queue()
            callback = lambda indata, frames, time_info, status: q.put(indata[:, 0].copy())

            with sd.InputStream(samplerate=sr, channels=1, dtype="float32", blocksize=block, device=mic, callback=callback):
                self.set_status("Calibrating the microphone (stay quiet for a second)...")
                ambient = [_rms(q.get(timeout=3)) for _ in range(10)]
                threshold = max(MIN_SPEECH_RMS, NOISE_MARGIN * float(np.median(ambient)))
                log(f"[voice] room noise {np.median(ambient):.4f}, speech threshold {threshold:.4f}")
                self.set_status("Listening. Say e.g. 'pick up the hammer'")

                pre = collections.deque(maxlen=3)                  # keep 0.3 s before the speech was detected
                buf, speaking, loud, quiet = [], False, 0, 0
                while not stop_event.is_set():
                    try:
                        blk = q.get(timeout=0.5)
                    except queue.Empty:
                        continue
                    r = _rms(blk)
                    if not speaking:
                        pre.append(blk)
                        loud = loud + 1 if r > threshold else 0
                        if loud >= 2:
                            speaking, buf, quiet = True, list(pre), 0
                            self.set_status("Hearing you...")
                    else:
                        buf.append(blk)
                        quiet = quiet + 1 if r < 0.7 * threshold else 0
                        if quiet >= end_blocks or len(buf) >= max_blocks:
                            audio = np.concatenate(buf)
                            speaking, buf, loud, quiet = False, [], 0, 0
                            pre.clear()
                            self._handle(audio, model, sr)
                            while not q.empty():                   # drop what was recorded while transcribing
                                q.get_nowait()
                            self.set_status("Listening. Say e.g. 'pick up the hammer'")
        except Exception as exc:
            log(f"[voice] error: {exc!r}")
        self.set_status("Voice input stopped.")

    def _handle(self, audio, model, sr):
        if len(audio) < 0.4 * sr:
            return
        if self.get_sim_state() != "idle":
            log("[voice] arm is busy: speech ignored")
            return
        self.set_status("Transcribing...")
        text = model.transcribe(audio, language="en", fp16=False, initial_prompt=WHISPER_PROMPT,
                                condition_on_previous_text=False)["text"].strip()
        if not text:
            return
        log(f'[voice] heard: "{text}"')
        kind, arg = parse_command(text)
        if kind in ("tool", "reset"):
            self.on_command(kind, arg, "voice")
        else:
            log("[voice] ignored: name exactly one tool (wrench, screwdriver, hammer, pliers)")


class Hold:
    """a gesture must be held for `need` seconds; a glitch shorter than `grace` does not reset the timer"""
    def __init__(self, grace):
        self.grace = grace
        self.reset()

    def reset(self):
        self.label, self.start, self.last_good = None, 0.0, 0.0

    def update(self, label, now, need):
        if label is not None and label == self.label:
            self.last_good = now
        elif label is not None:
            self.label, self.start, self.last_good = label, now, now
        elif self.label is not None and now - self.last_good > self.grace:
            self.reset()
        if self.label is None:
            return 0.0, False
        held = now - self.start
        return min(1.0, held / need), held >= need


class GestureInput:
    def __init__(self, on_command, set_frame, set_status, get_sim_state):
        self.on_command, self.set_frame, self.set_status, self.get_sim_state = on_command, set_frame, set_status, get_sim_state
        self.thread, self.stop_event = None, threading.Event()

    def start(self, camera_index):
        if self.thread is not None and self.thread.is_alive():
            return
        self.stop_event = threading.Event()
        self.thread = threading.Thread(target=self._run, args=(camera_index, self.stop_event), daemon=True)
        self.thread.start()

    def stop(self):
        self.stop_event.set()
        if self.thread is not None and self.thread.is_alive():
            self.thread.join(timeout=3)

    def _run(self, camera_index, stop_event):
        try:
            self._loop(camera_index, stop_event)
        except Exception as exc:
            log(f"[vision] error: {exc!r}")
        self.set_status("Vision stopped.", 0.0)

    def _loop(self, camera_index, stop_event):
        import cv2
        import mediapipe as mp
        from mediapipe.tasks.python import vision
        import gesture as G

        G.ensure_model()
        cap = cv2.VideoCapture(camera_index, G.CAMERA_BACKEND)
        if not cap.isOpened():
            log(f"[vision] could not open camera {camera_index}.")
            return
        log(f"[vision] camera {camera_index} open. Hold a gesture still for {HOLD_SEC}s: {GESTURE_TO_TOOL}")

        ARMED, LOCKED = "ARMED", "LOCKED"
        state, hold = ARMED, Hold(GRACE_SEC)
        prev_P, prev_t = None, None
        last_send, no_gesture_since = time.monotonic() - COOLDOWN_SEC, None
        last_ts, last_push, banner, banner_until = -1, 0.0, "", 0.0
        try:
            with vision.HandLandmarker.create_from_options(G.make_landmarker_options()) as landmarker:
                while not stop_event.is_set():
                    ok, frame = cap.read()
                    if not ok:
                        log("[vision] the camera returned no frame; stopping.")
                        break
                    frame = cv2.flip(frame, 1)
                    h, w = frame.shape[:2]
                    now = time.monotonic()
                    ts = int(now * 1000)
                    ts = ts if ts > last_ts else last_ts + 1
                    last_ts = ts
                    result = landmarker.detect_for_video(
                        mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)), ts)

                    gesture, still = None, False
                    if result.hand_landmarks:
                        world = np.array([[p.x, p.y, p.z] for p in result.hand_world_landmarks[0]])
                        P = np.array([[p.x * w, p.y * h] for p in result.hand_landmarks[0]])
                        G.draw_skeleton(frame, [(int(x), int(y)) for x, y in P])
                        gesture = G.classify_gesture(G.extract_features(world))
                        palm_px = G.dist(P, 0, 9) + 1e-9
                        if prev_P is not None and now > prev_t:
                            speed = float(np.mean(np.linalg.norm(P - prev_P, axis=1))) / palm_px / (now - prev_t)
                        else:
                            speed = float("inf")
                        prev_P, prev_t = P, now
                        still = (speed < MOTION_MAX_PER_SEC) or not REQUIRE_STILL
                    else:
                        prev_P, prev_t = None, None

                    busy = self.get_sim_state() != "idle"
                    progress, status = 0.0, ""
                    if state == ARMED:
                        if busy:                                    # the arm is working: lock until the hand is relaxed
                            state, no_gesture_since = LOCKED, None
                            hold.reset()
                        elif now - last_send < COOLDOWN_SEC:
                            hold.reset()
                            status = "Cooldown..."
                        elif result.hand_landmarks:
                            cand = gesture if (gesture in GESTURE_TO_TOOL and still) else None
                            progress, done = hold.update(cand, now, HOLD_SEC)
                            if done:
                                tool = GESTURE_TO_TOOL[cand]
                                log(f"[vision] gesture {cand} -> {tool}")
                                self.on_command("tool", tool, "vision")
                                state, last_send, no_gesture_since = LOCKED, now, None
                                banner, banner_until = f"SENT: {tool}", now + 1.5
                                hold.reset()
                            elif cand is not None:
                                status = f"Hold: {GESTURE_TO_TOOL[cand]}"
                            else:
                                status = "Show a tool gesture (1-4 fingers)"
                        else:
                            hold.update(None, now, HOLD_SEC)
                            status = "Show your hand"
                    if state == LOCKED:
                        hold.reset()
                        if busy:
                            status, no_gesture_since = "Arm busy - wait", None
                        else:
                            status = "Relax your hand to re-arm"
                            if gesture in GESTURE_TO_TOOL:
                                no_gesture_since = None
                            else:
                                no_gesture_since = no_gesture_since or now
                                if now - no_gesture_since >= REARM_SEC and now - last_send >= COOLDOWN_SEC:
                                    state = ARMED

                    if now - last_push > 0.07:                      # about 14 preview frames per second
                        if progress > 0:
                            cv2.rectangle(frame, (20, 20), (20 + int(300 * progress), 32), (0, 255, 0), -1)
                        cv2.putText(frame, banner if now < banner_until else status, (20, h - 20),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
                        ok_enc, buffer = cv2.imencode(".jpg", frame, [int(cv2.IMWRITE_JPEG_QUALITY), 70])
                        if ok_enc:
                            self.set_frame(buffer.tobytes())
                        self.set_status(status or state, progress)
                        last_push = now
        finally:
            cap.release()

## 5. Control panel
Run this cell last.

1. **Choose a method** (Text, Voice or Vision). Nothing else is enabled until you do.
2. Click **Start simulation**. The requirements for your method are checked first; if something is missing the log tells you what to install.
3. Wait for the viewer window and `READY`/`idle` in the log (loading the policy takes a minute on CPU).
4. Give instructions. You can switch methods at any time without restarting the simulation.

In [ ]:
import ipywidgets as W
from IPython.display import display

# ---------------- log window ----------------
LOG_LINES = []
log_box = W.Textarea(value="", disabled=True, layout=W.Layout(width="100%", height="240px"))
def ui_log(message):
    stamp = time.strftime("%H:%M:%S")
    for line in (str(message).splitlines() or [""]):
        LOG_LINES.append(f"{stamp}  {line}")
    del LOG_LINES[:-300]
    log_box.value = "\n".join(LOG_LINES)
_log_sink["fn"] = ui_log

# ---------------- state shown to the user ----------------
sim_status = W.HTML("<b>Simulation:</b> not running")
arm_status = W.HTML("<b>Arm:</b> -")
task_label = W.HTML("<b>Last command:</b> none yet")

def _on_sim_state(state):
    arm_status.value = f"<b>Arm:</b> {html.escape(state)}"
    if state == "idle":
        sim_status.value = "<b>Simulation:</b> running - waiting for a command"
    elif state in ("stopped",):
        sim_status.value = "<b>Simulation:</b> stopped"
sim = SimProcess(on_state=_on_sim_state)

# ---------------- the one entry point for every method ----------------
def submit(kind, arg, source):
    if not sim.running:
        log("The simulation is not running. Click 'Start simulation' first.")
        return False
    if kind == "reset":
        sim.send("reset")
        log(f"[{source}] reset requested")
        return True
    if sim.state != "idle":
        log(f"[{source}] the arm is busy ({sim.state}); '{arg}' ignored")
        return False
    sim.mark_busy()
    sim.send(arg)
    task_label.value = f"<b>Last command:</b> {html.escape(arg)} <i>(via {source})</i>"
    log(f"[{source}] sent: {arg}")
    return True

# ---------------- text panel ----------------
text_in = W.Text(placeholder="e.g. pick up the hammer", layout=W.Layout(width="55%"), continuous_update=False)
send_btn = W.Button(description="Send", button_style="primary", icon="paper-plane")
tool_buttons = []
for _tool in sim_server.TOOL_SYNONYMS:
    b = W.Button(description=_tool)
    b.on_click(lambda _b, t=_tool: submit("tool", t, "text"))
    tool_buttons.append(b)
def on_send(_=None):
    text, text_in.value = text_in.value, ""
    kind, arg = parse_command(text)
    if kind in ("tool", "reset"):
        submit(kind, arg, "text")
    elif text.strip():
        log("No tool found in that text. Name exactly one of: " + ", ".join(sim_server.TOOL_SYNONYMS))
send_btn.on_click(on_send)
try:
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        text_in.on_submit(on_send)                 # Enter key (older ipywidgets API)
except Exception:
    pass
text_panel = W.VBox([W.HBox([text_in, send_btn]), W.HBox([W.Label("or click:")] + tool_buttons)])

# ---------------- voice panel ----------------
def _input_devices():
    try:
        import sounddevice as sd
        return [(f"[{i}] {d['name']}", i) for i, d in enumerate(sd.query_devices()) if d["max_input_channels"] > 0]
    except Exception:
        return []
mic_dd = W.Dropdown(options=[("System default", None)] + _input_devices(), description="Microphone:", style={"description_width": "90px"})
model_dd = W.Dropdown(options=["tiny", "base", "small"], value=WHISPER_MODEL, description="Whisper:", style={"description_width": "90px"})
voice_status = W.Label(value="Voice is idle. It listens while the simulation is running.")
voice_panel = W.VBox([mic_dd, model_dd, voice_status])

# ---------------- vision panel ----------------
cam_in = W.IntText(value=CAMERA_INDEX, description="Camera:", style={"description_width": "90px"}, layout=W.Layout(width="200px"))
rows = "".join(f"<tr><td style='padding-right:14px'><b>{g} finger(s)</b></td><td>{t}</td></tr>" for g, t in GESTURE_TO_TOOL.items())
gesture_table = W.HTML(f"<table>{rows}</table><i>Hold the gesture still for about {HOLD_SEC} s, then relax your hand.</i>")
preview = W.Image(format="jpeg", width=480)
vision_progress = W.FloatProgress(value=0, min=0, max=1, description="Hold:", layout=W.Layout(width="300px"))
vision_status = W.Label(value="Vision is idle. It starts with the simulation.")
vision_panel = W.VBox([cam_in, gesture_table, vision_status, vision_progress, preview])

panels = {"text": text_panel, "voice": voice_panel, "vision": vision_panel}

# ---------------- wiring ----------------
def _command(kind, arg, source):
    submit(kind, arg, source)
voice = VoiceInput(_command, lambda s: setattr(voice_status, "value", s), lambda: sim.state)
gestures = GestureInput(_command, lambda b: setattr(preview, "value", b),
                        lambda text, p: (setattr(vision_status, "value", text), setattr(vision_progress, "value", p)),
                        lambda: sim.state)

def stop_inputs():
    voice.stop()
    gestures.stop()

def start_input():
    if method.value == "voice":
        voice.start(mic_dd.value, model_dd.value)
    elif method.value == "vision":
        gestures.start(int(cam_in.value))

# ---------------- the choice ----------------
method_prompt = W.HTML("<h3>1. How do you want to give instructions?</h3>")
method = W.ToggleButtons(options=[("Text", "text"), ("Voice (microphone)", "voice"), ("Vision (hand gestures)", "vision")],
                         index=None, style={"button_width": "190px"})
start_btn = W.Button(description="Start simulation", button_style="success", icon="play", disabled=True)
stop_btn = W.Button(description="Stop simulation", button_style="danger", icon="stop")
reset_btn = W.Button(description="Reset scene", icon="refresh")

def show_panel():
    for key, panel in panels.items():
        panel.layout.display = "" if key == method.value else "none"

def on_method_change(change):
    stop_inputs()
    show_panel()
    start_btn.disabled = False
    log(f"Method: {method.value}")
    if sim.running:
        problems, warnings = preflight(method.value, check_common=False)
        for msg in warnings:
            log("warning: " + msg)
        if problems:
            log("Cannot use this method yet:")
            for msg in problems:
                log("  - " + msg)
        else:
            start_input()

def _monitor():
    while sim.running:
        time.sleep(1)
    stop_inputs()
    sim_status.value = "<b>Simulation:</b> stopped"
    arm_status.value = "<b>Arm:</b> -"

def on_start(_):
    if method.value is None:
        log("Choose a method first (Text, Voice or Vision).")
        return
    if sim.running:
        log("The simulation is already running.")
        return
    problems, warnings = preflight(method.value)
    for msg in warnings:
        log("warning: " + msg)
    if problems:
        log("Cannot start. Fix this first:")
        for msg in problems:
            log("  - " + msg)
        return
    log("Starting the simulation (loading the policy takes a while)...")
    sim_status.value = "<b>Simulation:</b> starting..."
    sim.start()
    start_input()
    threading.Thread(target=_monitor, daemon=True).start()

def on_stop(_):
    stop_inputs()
    sim.stop()
    sim_status.value = "<b>Simulation:</b> stopped"

def on_reset(_):
    submit("reset", None, "button")

method.observe(on_method_change, names="value")
start_btn.on_click(on_start)
stop_btn.on_click(on_stop)
reset_btn.on_click(on_reset)
atexit.register(lambda: (stop_inputs(), sim.stop()))

show_panel()
display(W.VBox([
    method_prompt, method,
    W.HBox([start_btn, stop_btn, reset_btn]),
    sim_status, arm_status, task_label,
    panels["text"], panels["voice"], panels["vision"],
    W.HTML("<b>Log</b>"), log_box,
]))

## If something goes wrong

| Symptom | Cause and fix |
|---|---|
| "Cannot start" in the log | It lists exactly what is missing (checkpoint path, dataset `meta/`, packages, camera, busy port). Fix it and click Start again. |
| `UDP port 5005 is in use` | `deploy_policy.ipynb` or an old `sim_server.py` is still running. Stop it, or `pkill -f sim_server.py`. |
| No viewer window, `[sim] [error] ...` | Read the `[sim]` lines. Most common: wrong `CKPT` (must be the `pretrained_model` folder). For the full traceback run `python sim_server.py --ckpt <path>` in a terminal. |
| Command ignored: "arm is busy" | By design. The arm finishes, returns home, and then accepts the next command. |
| Command refused: "already in the green tray" | Send `reset` (or click **Reset scene**) to put the tools back. |
| Voice: nothing heard | Pick the right microphone; the log shows the measured room noise and threshold. Speak after "Listening". |
| Voice: heard but ignored | The sentence must name exactly one tool. Whisper heard a different word? Try `WHISPER_MODEL = "small"`. |
| Voice or vision breaks `cv2` / `numpy` after `pip install` | `mediapipe` can pull a second OpenCV package. `pip uninstall -y opencv-python opencv-contrib-python`, then `pip install --force-reinstall opencv-python-headless`. Restart the kernel. |
| Vision: camera cannot be opened | Change `CAMERA_INDEX`; close other programs that use the camera. On macOS use `cv2.CAP_ANY` in `gesture.py`. |
| Vision: wrong or flickering gesture | Run `python gesture.py` alone: it prints the finger-straightness numbers; tune `STRAIGHT_MIN` there. |

**Kernel restarts / closing the notebook:** the simulation window closes by itself when the notebook process disappears.
**Other programs** (`gesture_command.py --live`, `send_command.py`) can send the same commands to port 5005 while this simulation is running.